# 03 — Repetition-independent train/test split

**MATLAB equivalent:** `Split.m`

This is a **within-subject, repetition-independent** split:

- Training: repetitions 1–4
- Testing: repetitions 5–6

The split is performed after window generation, using each window's repetition label. Since an entire repetition belongs to only one partition, overlapping windows from a repetition cannot leak across train and test.

In [ ]:
from pathlib import Path
import numpy as np
import joblib

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_ROOT = PROJECT_ROOT / "data" / "NinaPro_DB5"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

N_SUBJECTS = 10
N_CHANNELS = 16
FS = 200
WINDOW_SAMPLES = 40
STEP_SAMPLES = 10
CLASS_ORDER = np.arange(13)  # 0 = rest, 1–12 = movements

WindowedDB5 = joblib.load(OUTPUT_DIR / "WindowedDB5.joblib")

TRAIN_REPETITIONS = np.arange(1, 5)
TEST_REPETITIONS = np.arange(5, 7)

## Subset helper

In [ ]:
def subset_windows(source, mask):
    """Subset every field in a Rest or Movement window store consistently."""
    mask = np.asarray(mask, dtype=bool)
    output = {}

    for key, values in source.items():
        if key == "EMG":
            output[key] = [window for window, keep in zip(values, mask) if keep]
        else:
            output[key] = np.asarray(values)[mask].copy()

    return output

def split_subject(subject_data):
    train = {}
    test = {}

    for segment_name in ("Rest", "Movement"):
        segment = subject_data[segment_name]
        repetitions = np.asarray(segment["RepetitionLabel"])

        train_mask = np.isin(repetitions, TRAIN_REPETITIONS)
        test_mask = np.isin(repetitions, TEST_REPETITIONS)

        train[segment_name] = subset_windows(segment, train_mask)
        test[segment_name] = subset_windows(segment, test_mask)

        # Integrity check: no repetition may appear in both partitions.
        train_reps = set(train[segment_name]["RepetitionLabel"].tolist())
        test_reps = set(test[segment_name]["RepetitionLabel"].tolist())
        assert train_reps.isdisjoint(test_reps)

    return train, test

## Split all subjects and save

In [ ]:
TrainDB = {}
TestDB = {}

for s in range(1, N_SUBJECTS + 1):
    subject_name = f"S{s}"
    TrainDB[subject_name], TestDB[subject_name] = split_subject(
        WindowedDB5[subject_name]
    )

    print(f"\\n{subject_name}")
    print(" Training rest:", len(TrainDB[subject_name]["Rest"]["EMG"]))
    print(" Training movement:", len(TrainDB[subject_name]["Movement"]["EMG"]))
    print(" Testing rest:", len(TestDB[subject_name]["Rest"]["EMG"]))
    print(" Testing movement:", len(TestDB[subject_name]["Movement"]["EMG"]))

joblib.dump({"TrainDB": TrainDB, "TestDB": TestDB},
            OUTPUT_DIR / "Train_Test_DB.joblib", compress=3)

print("\\nTrain/test splitting completed and saved.")